# Alzheimer Final v6 — v2 베이스 + 버그 픽스 + Threshold 최적화

## 변경점 (v2 대비)
1. **버그 픽스**: Phase 1과 Phase 2 통합해서 매 epoch best 추적 (총 4 epoch 중 최고 저장)
2. **Threshold 최적화**: Dirichlet random search 500회로 클래스별 임계값 최적화
3. **Label Smoothing 0.05** 추가 (cRT 단계의 CrossEntropy에)
4. **Epoch 약간 증가**: Phase 1을 3 → 4 epoch (조금 더 여유롭게)
5. **TTA 3-way**: original + flip + small rotation

## 유지 (v2 검증 설정)
- ConvNeXt-Tiny
- 224×224 정사각형
- 단일 LR = 3e-4
- CBFocalLoss + OneCycleLR
- cRT 1:1:1:1, 1 epoch
- 환자단위 4-Fold + 환자투표 + 슬라이스 가중치 1.5
- 중앙 30장만 학습 (15~44)

## 예상 시간
약 50~60분


In [1]:
import os, time, random, gc
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler

import timm
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score, classification_report

from PIL import Image
import torchvision.transforms as T

import warnings
warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


cuda NVIDIA GeForce RTX 5070


In [2]:
DATA_ROOT = Path(r"C:\\Users\\USER\\Desktop\\대학\\3학년 1학기\\머신러닝 및 실습\\캐글 알츠하이머 - 개인\\alzheimer-prediction")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR  = DATA_ROOT / "test"
TRAIN_CSV = DATA_ROOT / "train.csv"
SUB_CSV   = DATA_ROOT / "sample_submission.csv"

OUTPUT_DIR = Path("./outputs"); OUTPUT_DIR.mkdir(exist_ok=True)
CKPT_DIR = OUTPUT_DIR / "checkpoints_final6"; CKPT_DIR.mkdir(exist_ok=True)

LABEL2IDX = {'MildDemented': 0, 'ModerateDemented': 1, 'NonDemented': 2, 'VeryMildDemented': 3}
IDX2LABEL = {v: k for k, v in LABEL2IDX.items()}
NUM_CLASSES = 4
SLICES_PER_PATIENT = 61

# v2 검증 설정 (정사각형)
IMG_SIZE = 224
MODEL_NAME = 'convnext_tiny'
BATCH_SIZE = 64
NUM_EPOCHS = 4         # v2의 3 → 4 (한 epoch 더 여유)
NUM_EPOCHS_CRT = 1
LR_MAX = 3e-4
LR_CRT = 1e-4
WEIGHT_DECAY = 0.1
NUM_FOLDS = 4
NUM_WORKERS = 0
DROP_RATE = 0.4
DROP_PATH_RATE = 0.3
LABEL_SMOOTHING = 0.05  # cRT 단계에서만

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]


In [3]:
def get_patient_id(fn):
    num = int(fn.replace('train_', '').replace('test_', '').replace('.jpg', ''))
    return (num - 1) // SLICES_PER_PATIENT

def get_slice_idx(fn):
    num = int(fn.replace('train_', '').replace('test_', '').replace('.jpg', ''))
    return (num - 1) % SLICES_PER_PATIENT

train_df_full = pd.read_csv(TRAIN_CSV)
train_df_full['label_idx'] = train_df_full['label'].map(LABEL2IDX)
train_df_full['patient_id'] = train_df_full['filename'].apply(get_patient_id)
train_df_full['slice_idx']  = train_df_full['filename'].apply(get_slice_idx)

train_df = train_df_full[(train_df_full['slice_idx'] >= 15) & (train_df_full['slice_idx'] < 45)].reset_index(drop=True)

sub_df = pd.read_csv(SUB_CSV)
sub_df['patient_id'] = sub_df['filename'].apply(get_patient_id)
sub_df['slice_idx']  = sub_df['filename'].apply(get_slice_idx)

print(f"Train(full): {len(train_df_full)} → Train(filtered): {len(train_df)}")
print(f"Test: {len(sub_df)}")
print(train_df['label'].value_counts())

class_counts = train_df['label_idx'].value_counts().sort_index().values
print(f"class_counts: {class_counts}")


Train(full): 68808 → Train(filtered): 33840
Test: 17629
label
NonDemented         26400
VeryMildDemented     5340
MildDemented         1980
ModerateDemented      120
Name: count, dtype: int64
class_counts: [ 1980   120 26400  5340]


In [4]:
class AlzheimerDataset(Dataset):
    def __init__(self, df, img_dir, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.img_dir = Path(img_dir)
        self.transform = transform
        self.is_test = is_test
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(self.img_dir / row['filename']).convert('RGB')
        if self.transform:
            img = self.transform(img)
        if self.is_test:
            return img
        return img, int(row['label_idx'])

def get_train_transform():
    return T.Compose([
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomAffine(degrees=10, translate=(0.05, 0.05), scale=(0.95, 1.05)),
        T.ColorJitter(brightness=0.1, contrast=0.1),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
        T.RandomErasing(p=0.25, scale=(0.02, 0.1), ratio=(0.3, 3.3), value=0),
    ])

def get_val_transform():
    return T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

# TTA 3-way
def get_tta_original():
    return T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])

def get_tta_flip():
    return T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.RandomHorizontalFlip(p=1.0),
                      T.ToTensor(), T.Normalize(MEAN, STD)])

def get_tta_rotate():
    return T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)),
                      T.RandomAffine(degrees=(5, 5), translate=(0, 0), scale=(1.0, 1.0)),
                      T.ToTensor(), T.Normalize(MEAN, STD)])


In [5]:
class CBFocalLoss(nn.Module):
    def __init__(self, class_counts, beta=0.9999, gamma=2.0):
        super().__init__()
        cc = np.asarray(class_counts, dtype=np.float64)
        eff = 1.0 - np.power(beta, cc)
        w = (1.0 - beta) / eff
        w = w / w.sum() * len(cc)
        self.register_buffer('weights', torch.tensor(w, dtype=torch.float32))
        self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        focal = (1 - pt) ** self.gamma * ce
        return (self.weights[targets] * focal).mean()

def build_model():
    m = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES,
                          drop_rate=DROP_RATE, drop_path_rate=DROP_PATH_RATE)
    return m.to(device)

def freeze_backbone(model):
    for name, p in model.named_parameters():
        p.requires_grad = ('head' in name)


In [6]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler):
    model.train()
    tl, tn = 0.0, 0
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast():
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        if scheduler is not None:
            scheduler.step()
        bs = imgs.size(0)
        tl += loss.item() * bs
        tn += bs
    return tl / tn

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    AL, AY = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast():
            logits = model(imgs)
        AL.append(logits.float().cpu().numpy())
        AY.append(labels.numpy())
    L = np.concatenate(AL, 0); Y = np.concatenate(AY, 0)
    P = L.argmax(1)
    return f1_score(Y, P, average='macro'), L, Y, P

@torch.no_grad()
def predict_logits(model, loader):
    model.eval()
    A = []
    for imgs in loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast():
            logits = model(imgs)
        A.append(logits.float().cpu().numpy())
    return np.concatenate(A, 0)


In [7]:
sgkf = StratifiedGroupKFold(n_splits=NUM_FOLDS, shuffle=True, random_state=SEED)
folds = []
for tr_idx, va_idx in sgkf.split(train_df, train_df['label_idx'], groups=train_df['patient_id']):
    folds.append((tr_idx, va_idx))

for fi, (tr, va) in enumerate(folds):
    trp = set(train_df.iloc[tr]['patient_id'])
    vap = set(train_df.iloc[va]['patient_id'])
    print(f"Fold {fi}: tr={len(tr)} va={len(va)} overlap={len(trp & vap)}")


Fold 0: tr=25380 va=8460 overlap=0
Fold 1: tr=25380 va=8460 overlap=0
Fold 2: tr=25380 va=8460 overlap=0
Fold 3: tr=25380 va=8460 overlap=0


In [8]:
# ========================================
# 메인 학습 루프
# ========================================
oof_logits = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)
oof_labels = np.zeros(len(train_df), dtype=np.int64)
fold_scores = []
t0 = time.time()

for fi, (tr_idx, va_idx) in enumerate(folds):
    print(f"\n=== FOLD {fi+1}/{NUM_FOLDS} ===")
    tf = time.time()
    tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
    va_df = train_df.iloc[va_idx].reset_index(drop=True)

    tr_ds = AlzheimerDataset(tr_df, TRAIN_DIR, transform=get_train_transform())
    va_ds = AlzheimerDataset(va_df, TRAIN_DIR, transform=get_val_transform())
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                           num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)
    criterion = CBFocalLoss(class_counts).to(device)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=LR_MAX, epochs=NUM_EPOCHS, steps_per_epoch=len(tr_loader),
        pct_start=0.1, anneal_strategy='cos')
    scaler = GradScaler()

    # ★ 버그 픽스: best 추적을 Phase 1부터 시작
    best_f1, best_state = -1.0, None
    best_L, best_Y = None, None

    print(">> Phase 1: CBFocalLoss + OneCycleLR")
    for ep in range(NUM_EPOCHS):
        te = time.time()
        tl = train_one_epoch(model, tr_loader, criterion, optimizer, scheduler, scaler)
        vf, L, Y, P = evaluate(model, va_loader)
        marker = ""
        if vf > best_f1:
            best_f1 = vf
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_L, best_Y = L, Y
            marker = " *"
        print(f"  Ep {ep+1}/{NUM_EPOCHS} loss={tl:.4f} val_f1={vf:.4f} ({time.time()-te:.1f}s){marker}")

    print(">> Phase 2: cRT 1:1:1:1 + Label Smoothing")
    freeze_backbone(model)
    tr_labels = tr_df['label_idx'].values
    cls_cnt = np.bincount(tr_labels, minlength=NUM_CLASSES)
    sw = np.array([1.0 / cls_cnt[y] for y in tr_labels], dtype=np.float64)
    sampler = WeightedRandomSampler(sw, num_samples=len(tr_labels), replacement=True)
    tr_loader_bal = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler,
                               num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
    head_params = [p for p in model.parameters() if p.requires_grad]
    optimizer2 = torch.optim.AdamW(head_params, lr=LR_CRT, weight_decay=WEIGHT_DECAY)
    criterion2 = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

    for ep in range(NUM_EPOCHS_CRT):
        te = time.time()
        tl = train_one_epoch(model, tr_loader_bal, criterion2, optimizer2, None, scaler)
        vf, L, Y, P = evaluate(model, va_loader)
        marker = ""
        if vf > best_f1:  # ★ 버그 픽스: Phase 1 best도 비교 대상
            best_f1 = vf
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_L, best_Y = L, Y
            marker = " *"
        print(f"  cRT Ep {ep+1}/{NUM_EPOCHS_CRT} loss={tl:.4f} val_f1={vf:.4f} ({time.time()-te:.1f}s){marker}")

    oof_logits[va_idx] = best_L
    oof_labels[va_idx] = best_Y
    torch.save(best_state, CKPT_DIR / f"fold{fi}_best.pt")
    fold_scores.append(best_f1)
    print(f"Fold {fi} BEST={best_f1:.4f} ({time.time()-tf:.1f}s)")

    del model, optimizer, optimizer2, scheduler, scaler, tr_loader, va_loader, tr_loader_bal
    torch.cuda.empty_cache(); gc.collect()

print(f"\nTotal: {(time.time()-t0)/60:.1f}min")
print(f"Folds: {[f'{s:.4f}' for s in fold_scores]}")
print(f"Mean: {np.mean(fold_scores):.4f}  Std: {np.std(fold_scores):.4f}")



=== FOLD 1/4 ===
>> Phase 1: CBFocalLoss + OneCycleLR
  Ep 1/4 loss=0.0597 val_f1=0.4947 (130.9s) *
  Ep 2/4 loss=0.0260 val_f1=0.6989 (114.8s) *
  Ep 3/4 loss=0.0127 val_f1=0.8379 (109.5s) *
  Ep 4/4 loss=0.0070 val_f1=0.8855 (109.6s) *
>> Phase 2: cRT 1:1:1:1 + Label Smoothing
  cRT Ep 1/1 loss=0.4553 val_f1=0.8510 (85.0s)
Fold 0 BEST=0.8855 (550.5s)

=== FOLD 2/4 ===


>> Phase 1: CBFocalLoss + OneCycleLR
  Ep 1/4 loss=0.0604 val_f1=0.5282 (110.5s) *
  Ep 2/4 loss=0.0267 val_f1=0.7184 (110.8s) *
  Ep 3/4 loss=0.0133 val_f1=0.8257 (109.4s) *
  Ep 4/4 loss=0.0073 val_f1=0.8695 (109.5s) *
>> Phase 2: cRT 1:1:1:1 + Label Smoothing
  cRT Ep 1/1 loss=0.4668 val_f1=0.8361 (84.5s)
Fold 1 BEST=0.8695 (525.2s)

=== FOLD 3/4 ===
>> Phase 1: CBFocalLoss + OneCycleLR
  Ep 1/4 loss=0.0590 val_f1=0.4017 (110.8s) *
  Ep 2/4 loss=0.0271 val_f1=0.6312 (111.1s) *
  Ep 3/4 loss=0.0136 val_f1=0.7871 (109.7s) *
  Ep 4/4 loss=0.0080 val_f1=0.8127 (109.5s) *
>> Phase 2: cRT 1:1:1:1 + Label Smoothing
  cRT Ep 1/1 loss=0.4737 val_f1=0.7708 (83.9s)
Fold 2 BEST=0.8127 (525.6s)

=== FOLD 4/4 ===
>> Phase 1: CBFocalLoss + OneCycleLR
  Ep 1/4 loss=0.0721 val_f1=0.2191 (110.5s) *
  Ep 2/4 loss=0.0618 val_f1=0.2191 (110.7s)
  Ep 3/4 loss=0.0584 val_f1=0.2191 (110.7s)
  Ep 4/4 loss=0.0571 val_f1=0.2191 (110.7s)
>> Phase 2: cRT 1:1:1:1 + Label Smoothing
  cRT Ep 1/1 loss=1.3950 val_f1

In [14]:
# ========================================
# Fold 3 (Fold 4) 재학습 — LR 1e-4로 낮춤
# ========================================
FOLD_RETRY = 3  # 0-indexed
LR_RETRY = 1e-4  # 3e-4 → 1e-4로 낮춤

tr_idx, va_idx = folds[FOLD_RETRY]
tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
va_df = train_df.iloc[va_idx].reset_index(drop=True)

tr_ds = AlzheimerDataset(tr_df, TRAIN_DIR, transform=get_train_transform())
va_ds = AlzheimerDataset(va_df, TRAIN_DIR, transform=get_val_transform())
tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE*2, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True)

model = build_model()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_RETRY, weight_decay=WEIGHT_DECAY)
criterion = CBFocalLoss(class_counts).to(device)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR_RETRY, epochs=NUM_EPOCHS, steps_per_epoch=len(tr_loader),
    pct_start=0.1, anneal_strategy='cos')
scaler = GradScaler()

best_f1, best_state = -1.0, None
best_L, best_Y = None, None

print(f">> Fold {FOLD_RETRY} 재학습 (LR={LR_RETRY})")
print(">> Phase 1")
for ep in range(NUM_EPOCHS):
    te = time.time()
    tl = train_one_epoch(model, tr_loader, criterion, optimizer, scheduler, scaler)
    vf, L, Y, P = evaluate(model, va_loader)
    marker = ""
    if vf > best_f1:
        best_f1 = vf
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_L, best_Y = L, Y
        marker = " *"
    print(f"  Ep {ep+1}/{NUM_EPOCHS} loss={tl:.4f} val_f1={vf:.4f} ({time.time()-te:.1f}s){marker}")

print(">> Phase 2: cRT 1:1:1:1 + Label Smoothing")
freeze_backbone(model)
tr_labels = tr_df['label_idx'].values
cls_cnt = np.bincount(tr_labels, minlength=NUM_CLASSES)
sw = np.array([1.0 / cls_cnt[y] for y in tr_labels], dtype=np.float64)
sampler = WeightedRandomSampler(sw, num_samples=len(tr_labels), replacement=True)
tr_loader_bal = DataLoader(tr_ds, batch_size=BATCH_SIZE, sampler=sampler,
                           num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
head_params = [p for p in model.parameters() if p.requires_grad]
optimizer2 = torch.optim.AdamW(head_params, lr=LR_CRT, weight_decay=WEIGHT_DECAY)
criterion2 = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

for ep in range(NUM_EPOCHS_CRT):
    te = time.time()
    tl = train_one_epoch(model, tr_loader_bal, criterion2, optimizer2, None, scaler)
    vf, L, Y, P = evaluate(model, va_loader)
    marker = ""
    if vf > best_f1:
        best_f1 = vf
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_L, best_Y = L, Y
        marker = " *"
    print(f"  cRT Ep {ep+1}/{NUM_EPOCHS_CRT} loss={tl:.4f} val_f1={vf:.4f} ({time.time()-te:.1f}s){marker}")

# OOF 업데이트 + 체크포인트 덮어쓰기
oof_logits[va_idx] = best_L
oof_labels[va_idx] = best_Y
torch.save(best_state, CKPT_DIR / f"fold{FOLD_RETRY}_best.pt")
fold_scores[FOLD_RETRY] = best_f1
print(f"\nFold {FOLD_RETRY} BEST={best_f1:.4f}")
print(f"\n새 Fold 점수: {[f'{s:.4f}' for s in fold_scores]}")
print(f"새 Mean: {np.mean(fold_scores):.4f}  Std: {np.std(fold_scores):.4f}")

del model, optimizer, optimizer2, scheduler, scaler, tr_loader, va_loader, tr_loader_bal
torch.cuda.empty_cache(); gc.collect()

>> Fold 3 재학습 (LR=0.0001)
>> Phase 1
  Ep 1/4 loss=0.0487 val_f1=0.6324 (111.8s) *
  Ep 2/4 loss=0.0137 val_f1=0.8311 (115.7s) *
  Ep 3/4 loss=0.0046 val_f1=0.9361 (110.8s) *
  Ep 4/4 loss=0.0019 val_f1=0.9445 (110.8s) *
>> Phase 2: cRT 1:1:1:1 + Label Smoothing
  cRT Ep 1/1 loss=0.3128 val_f1=0.9208 (84.6s)

Fold 3 BEST=0.9445

새 Fold 점수: ['0.8855', '0.8695', '0.8127', '0.9445']
새 Mean: 0.8780  Std: 0.0469


8

In [15]:
# ========================================
# OOF 평가
# ========================================
oof_preds = oof_logits.argmax(1)
oof_f1 = f1_score(oof_labels, oof_preds, average='macro')
print(f"OOF Macro F1 (slice, argmax): {oof_f1:.4f}")
print(classification_report(oof_labels, oof_preds,
      target_names=[IDX2LABEL[i] for i in range(NUM_CLASSES)], digits=4))

oof_softmax = F.softmax(torch.from_numpy(oof_logits), dim=1).numpy()


OOF Macro F1 (slice, argmax): 0.8758
                  precision    recall  f1-score   support

    MildDemented     0.8659    0.9263    0.8951      1980
ModerateDemented     1.0000    1.0000    1.0000       120
     NonDemented     0.9584    0.9007    0.9287     26400
VeryMildDemented     0.6069    0.7721    0.6796      5340

        accuracy                         0.8822     33840
       macro avg     0.8578    0.8998    0.8758     33840
    weighted avg     0.8977    0.8822    0.8877     33840



In [16]:
# ========================================
# Threshold 최적화 (Dirichlet random search 500회)
# OOF softmax → 클래스별 임계값으로 Macro F1 극대화
# ========================================
def optimize_thresholds(probs, true_labels, n_search=500, seed=42):
    rng = np.random.RandomState(seed)
    best_f1 = 0.0
    best_thresh = np.ones(NUM_CLASSES) * 0.25

    # 균등 임계값(현재 argmax)의 F1
    preds_uniform = probs.argmax(axis=1)
    f1_uniform = f1_score(true_labels, preds_uniform, average='macro')
    print(f"  baseline (argmax) Macro F1: {f1_uniform:.4f}")
    best_f1 = f1_uniform

    for it in range(n_search):
        # Dirichlet으로 합이 1인 4차원 가중치 샘플
        thresh = rng.dirichlet(np.ones(NUM_CLASSES))
        adjusted = probs / (thresh + 1e-9)
        preds = adjusted.argmax(axis=1)
        f1 = f1_score(true_labels, preds, average='macro', zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = thresh.copy()

    print(f"  optimized Macro F1: {best_f1:.4f}  (+{best_f1 - f1_uniform:.4f})")
    print(f"  threshold: {dict(zip([IDX2LABEL[i] for i in range(NUM_CLASSES)], best_thresh.round(4)))}")
    return best_thresh, best_f1

print("=== Threshold Optimization (slice level) ===")
best_thresh_slice, best_f1_slice = optimize_thresholds(oof_softmax, oof_labels, n_search=500)

# 환자단위 OOF
oof_df = train_df.copy()
for c in range(NUM_CLASSES):
    oof_df[f'p{c}'] = oof_softmax[:, c]
pat = oof_df.groupby('patient_id')[[f'p{c}' for c in range(NUM_CLASSES)]].mean()
pat_probs = pat.values
pat_true = oof_df.groupby('patient_id')['label_idx'].first().values

# 환자단위 OOF에서도 threshold 최적화
print("\n=== Threshold Optimization (patient level) ===")
best_thresh_patient, best_f1_patient = optimize_thresholds(pat_probs, pat_true, n_search=500)


=== Threshold Optimization (slice level) ===
  baseline (argmax) Macro F1: 0.8758
  optimized Macro F1: 0.8785  (+0.0027)
  threshold: {'MildDemented': np.float64(0.4148), 'ModerateDemented': np.float64(0.1823), 'NonDemented': np.float64(0.1838), 'VeryMildDemented': np.float64(0.2191)}

=== Threshold Optimization (patient level) ===
  baseline (argmax) Macro F1: 0.9119
  optimized Macro F1: 0.9119  (+0.0000)
  threshold: {'MildDemented': np.float64(0.25), 'ModerateDemented': np.float64(0.25), 'NonDemented': np.float64(0.25), 'VeryMildDemented': np.float64(0.25)}


In [17]:
# ========================================
# Test 예측 + TTA 3-way
# ========================================
def slice_weight(s):
    return 1.5 if 15 <= s <= 45 else 1.0

tta_transforms = [
    ('original', get_tta_original()),
    ('flip',     get_tta_flip()),
    ('rotate',   get_tta_rotate()),
]

test_sm_sum = np.zeros((len(sub_df), NUM_CLASSES), dtype=np.float32)
n_predictions = 0

# loader 미리 생성
test_loaders = []
for tta_name, tfm in tta_transforms:
    ds = AlzheimerDataset(sub_df, TEST_DIR, transform=tfm, is_test=True)
    ld = DataLoader(ds, batch_size=BATCH_SIZE*2, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)
    test_loaders.append((tta_name, ld))

for fi in range(NUM_FOLDS):
    print(f"Predict fold {fi}")
    model = build_model()
    model.load_state_dict(torch.load(CKPT_DIR / f"fold{fi}_best.pt", map_location=device))

    for tta_name, ld in test_loaders:
        L = predict_logits(model, ld)
        sm = F.softmax(torch.from_numpy(L), dim=1).numpy()
        test_sm_sum += sm
        n_predictions += 1

    del model; torch.cuda.empty_cache(); gc.collect()

test_sm = test_sm_sum / n_predictions
print(f"\nTotal predictions: {n_predictions}")
print(f"slice argmax dist: {Counter(test_sm.argmax(1))}")


Predict fold 0
Predict fold 1
Predict fold 2
Predict fold 3

Total predictions: 12
slice argmax dist: Counter({np.int64(2): 12685, np.int64(3): 3828, np.int64(0): 1115, np.int64(1): 1})


In [18]:
# ========================================
# 환자단위 Soft Voting + Threshold 적용
# 4가지 버전 모두 생성해서 비교
# ========================================
tp = sub_df.copy()
for c in range(NUM_CLASSES):
    tp[f'p{c}'] = test_sm[:, c]
tp['w'] = tp['slice_idx'].apply(slice_weight)
for c in range(NUM_CLASSES):
    tp[f'wp{c}'] = tp[f'p{c}'] * tp['w']

# 환자별 가중 평균
cols = [f'wp{c}' for c in range(NUM_CLASSES)]
pat_p_test = tp.groupby('patient_id')[cols].sum()
pat_w_test = tp.groupby('patient_id')['w'].sum()
pat_p_test = pat_p_test.div(pat_w_test, axis=0).values  # (N_patients, 4)

# 4가지 예측
# (1) patient + argmax (= 기존 v2 방식)
pred_v1 = pat_p_test.argmax(axis=1)

# (2) patient + threshold (환자 OOF로 최적화한 threshold)
pred_v2 = (pat_p_test / (best_thresh_patient + 1e-9)).argmax(axis=1)

# (3) slice + argmax
pred_slice_v1 = test_sm.argmax(axis=1)

# (4) slice + threshold (slice OOF로 최적화한 threshold)
pred_slice_v2 = (test_sm / (best_thresh_slice + 1e-9)).argmax(axis=1)

print("(1) patient + argmax:    ", Counter(pred_v1))
print("(2) patient + threshold: ", Counter(pred_v2))
print("(3) slice + argmax:      ", Counter(pred_slice_v1))
print("(4) slice + threshold:   ", Counter(pred_slice_v2))


(1) patient + argmax:     Counter({np.int64(2): 209, np.int64(3): 67, np.int64(0): 13})
(2) patient + threshold:  Counter({np.int64(2): 209, np.int64(3): 67, np.int64(0): 13})
(3) slice + argmax:       Counter({np.int64(2): 12685, np.int64(3): 3828, np.int64(0): 1115, np.int64(1): 1})
(4) slice + threshold:    Counter({np.int64(2): 13336, np.int64(3): 3676, np.int64(0): 595, np.int64(1): 22})


In [19]:
# ========================================
# Submission 4종 저장
# ========================================
sample_sub = pd.read_csv(SUB_CSV)
pat_ids = sub_df.groupby('patient_id').first().index.values
pid2lbl_v1 = dict(zip(pat_ids, pred_v1))
pid2lbl_v2 = dict(zip(pat_ids, pred_v2))

# (1) patient + argmax
tp['final_v1'] = tp['patient_id'].map(pid2lbl_v1)
tp['label_v1'] = tp['final_v1'].map(IDX2LABEL)
sub1 = sample_sub[['filename']].merge(tp[['filename', 'label_v1']].rename(columns={'label_v1': 'label'}), on='filename', how='left')
sub1.to_csv(OUTPUT_DIR / "submission_Final_6_patient_argmax.csv", index=False)
print(f"(1) saved: submission_Final_6_patient_argmax.csv  | {dict(sub1['label'].value_counts())}")

# (2) patient + threshold ← 가장 기대되는 것
tp['final_v2'] = tp['patient_id'].map(pid2lbl_v2)
tp['label_v2'] = tp['final_v2'].map(IDX2LABEL)
sub2 = sample_sub[['filename']].merge(tp[['filename', 'label_v2']].rename(columns={'label_v2': 'label'}), on='filename', how='left')
sub2.to_csv(OUTPUT_DIR / "submission_Final_6_patient_threshold.csv", index=False)
print(f"(2) saved: submission_Final_6_patient_threshold.csv  | {dict(sub2['label'].value_counts())}")

# (3) slice + argmax
sl3 = [IDX2LABEL[i] for i in pred_slice_v1]
sub3 = pd.DataFrame({'filename': sub_df['filename'], 'label': sl3})
sub3 = sample_sub[['filename']].merge(sub3, on='filename', how='left')
sub3.to_csv(OUTPUT_DIR / "submission_Final_6_slice_argmax.csv", index=False)
print(f"(3) saved: submission_Final_6_slice_argmax.csv  | {dict(sub3['label'].value_counts())}")

# (4) slice + threshold
sl4 = [IDX2LABEL[i] for i in pred_slice_v2]
sub4 = pd.DataFrame({'filename': sub_df['filename'], 'label': sl4})
sub4 = sample_sub[['filename']].merge(sub4, on='filename', how='left')
sub4.to_csv(OUTPUT_DIR / "submission_Final_6_slice_threshold.csv", index=False)
print(f"(4) saved: submission_Final_6_slice_threshold.csv  | {dict(sub4['label'].value_counts())}")

print("\n=== 제출 추천 순서 ===")
print("1. submission_Final_6_patient_threshold.csv  ← 가장 기대됨")
print("2. submission_Final_6_patient_argmax.csv     ← v2와 직접 비교")
print("3. submission_Final_6_slice_threshold.csv")
print("4. submission_Final_6_slice_argmax.csv")


(1) saved: submission_Final_6_patient_argmax.csv  | {'NonDemented': np.int64(12749), 'VeryMildDemented': np.int64(4087), 'MildDemented': np.int64(793)}
(2) saved: submission_Final_6_patient_threshold.csv  | {'NonDemented': np.int64(12749), 'VeryMildDemented': np.int64(4087), 'MildDemented': np.int64(793)}
(3) saved: submission_Final_6_slice_argmax.csv  | {'NonDemented': np.int64(12685), 'VeryMildDemented': np.int64(3828), 'MildDemented': np.int64(1115), 'ModerateDemented': np.int64(1)}
(4) saved: submission_Final_6_slice_threshold.csv  | {'NonDemented': np.int64(13336), 'VeryMildDemented': np.int64(3676), 'MildDemented': np.int64(595), 'ModerateDemented': np.int64(22)}

=== 제출 추천 순서 ===
1. submission_Final_6_patient_threshold.csv  ← 가장 기대됨
2. submission_Final_6_patient_argmax.csv     ← v2와 직접 비교
3. submission_Final_6_slice_threshold.csv
4. submission_Final_6_slice_argmax.csv
